# 02.1 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/02-regression/01-what-is-a-model/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd
five = pd.read_csv(data_path("shopwave_visits_spend_5.csv"))
five

,customer_id,visits_last_month,spend_next_month
0,C001,2,50
1,C002,4,80
2,C003,6,90
3,C004,8,130
4,C005,10,150


## Exercise 1

| $x$ | $y$ | $\hat y = 15x$ | $\lvert y-\hat y\rvert$ |
|---|---|---|---|
| 2 | 50 | 30 | 20 |
| 4 | 80 | 60 | 20 |
| 6 | 90 | 90 | 0 |
| 8 | 130 | 120 | 10 |
| 10 | 150 | 150 | 0 |
| | | sum | **50** |

MAE = 50/5 = **£10**. Better than A (£32) and B (£40). Same shape as B, a better knob.

In [2]:
y = five["spend_next_month"]; x = five["visits_last_month"]
mae = lambda y, y_hat: (y - y_hat).abs().mean()
print("MAE C =", mae(y, 15 * x))

MAE C = 10.0


## Exercise 2

In [3]:
import numpy as np
best = (None, None, np.inf)
for k in np.arange(5, 20.5, 0.5):
    for c in range(0, 51):
        m = mae(y, k * x + c)
        if m < best[2]:
            best = (k, c, m)
print(f"best slope {best[0]}, intercept {best[1]}, MAE £{best[2]:.2f}")
print(f"least squares (12.5, 25): MAE £{mae(y, 12.5 * x + 25):.2f}")

best slope 12.5, intercept 25, MAE £4.00
least squares (12.5, 25): MAE £4.00


On this table the grid lands on the least-squares line, (12.5, 25), at MAE £4.00, but it is not alone: several neighbouring (slope, intercept) pairs tie at exactly £4.00. MAE has a flat bottom; squared error (02.2, 02.3) has one unique minimum, which is one reason it is the default for training.

## Exercise 3

In [4]:
cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
y20 = cust20["spend_next_month"]
ks = np.arange(0, 40.25, 0.25)
for col in ["visits_last_month", "items_last_month", "days_since_last"]:
    maes = [mae(y20, k * cust20[col]) for k in ks]
    print(f"{col:<18} best k = {ks[int(np.argmin(maes))]:>6}   MAE = £{min(maes):.2f}")

visits_last_month  best k =  19.25   MAE = £11.24
items_last_month   best k =  30.25   MAE = £20.11
days_since_last    best k =    4.5   MAE = £96.33


`visits_last_month` wins, then `items`. `days_since_last` is nearly useless as `k × days` because its relationship with spend is *negative* and needs an intercept to express (more days since last order, less spend). A one-knob model without an intercept cannot represent that at all.

## Exercise 4

Any three of:

- `spend_next_quarter` / `came_back`: the target itself.
- `recency_days` (01.9): computed after the target window; contains the answer.
- `Invoice`, `StockCode`, `Customer ID`: identifiers, not measurements. A model that "learns" customer 17850 spends a lot has memorised, not learned.
- `InvoiceDate` values after the cut-off: anything from the future.
- `Description`: free text; usable only after Module 09 turns it into numbers, and only for rows before the cut-off.